# Shipment Analytics — FreightFox Take-Home Assignment

This notebook explores `data/shipments.csv`, documents data quality issues found,
and answers the 5 business questions with the underlying query/calculation for each.

All cleaning logic lives in `clean_data.py` so the Streamlit dashboard uses the
exact same definitions as this notebook — no drift between "the analysis" and "the app."


In [1]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, binomtest
import sys
sys.path.append('..')
from clean_data import load_and_clean

pd.set_option('display.width', 150)
pd.set_option('display.max_columns', None)

df, quality_report = load_and_clean(path='../data/shipments.csv')
print(f"Rows after cleaning: {len(df)}")
df.head()


Rows after cleaning: 5000


,shipment_id,booking_date,pickup_date,delivery_date,origin_city,destination_city,region,mode,carrier_id,customer_id,promised_delivery_date,actual_delivery_date,freight_cost,distance_km,status,delay_days,flag_impossible_dates,flag_missing_actual_when_completed,is_late_by_date,valid_for_delay_analysis,cost_per_km
0,SHP003604,2026-04-18,2026-04-21,2026-04-27,Chandigarh,Guwahati,North,PTL,CARR_07,CUST_085,2026-04-27,2026-04-29,72921.16,894,Delivered,2.0,False,False,1.0,True,81.567293
1,SHP003362,2026-06-01,2026-06-01,2026-06-04,Kolkata,Raipur,East,FTL,CARR_03,CUST_108,2026-06-04,2026-06-03,2451.90,91,Delivered,-1.0,False,False,0.0,True,26.943956
2,SHP000796,2026-01-17,2026-01-20,2026-01-23,Surat,Jaipur,West,FTL,CARR_05,CUST_108,2026-01-23,NaT,10320.66,395,In-Transit,NaN,False,False,NaN,False,26.128253
3,SHP000291,2026-04-25,2026-04-25,2026-05-01,Nagpur,Bhubaneswar,Central,FTL,CARR_09,CUST_063,2026-05-01,2026-05-02,30242.52,1223,Delayed,1.0,False,False,1.0,True,24.728144
4,SHP003739,2026-01-29,2026-01-30,2026-02-03,Ahmedabad,Hyderabad,West,LTL,CARR_04,CUST_064,2026-02-03,2026-02-01,18434.26,1664,Delivered,-2.0,False,False,0.0,True,11.078281


## Data Quality — Overview

Before trusting any downstream number, here's what was found in the raw data
and how each issue was handled. This report is generated directly by
`clean_data.load_and_clean()` so it can never drift out of sync with the actual cleaning logic.


In [2]:
for k, v in quality_report.items():
    print(f"{k}: {v}")


n_raw_rows: 5015
n_exact_duplicates_dropped: 15
n_after_dedup: 5000
delivery_date_equals_promised_pct: 100.0
n_impossible_date_rows: 74
n_completed_missing_actual_date: 682
n_status_vs_date_mismatch: 1742
n_missing_booking_date: 71
n_missing_pickup_date: 87
n_valid_for_delay_analysis: 3444
n_origin_eq_destination: 244


### Key data quality findings

1. **15 exact duplicate rows** — same `shipment_id`, every column identical. Dropped.

2. **`delivery_date` is a fully redundant column.** It equals `promised_delivery_date`
   in 100% of rows (mean and std of the difference are both exactly 0). It is *not*
   a record of what actually happened, despite the name. We ignore it entirely and
   use `actual_delivery_date` as the only source of truth for what really occurred.

3. **The `status` field disagrees with date-derived reality in 1,742 of 5,000 rows (35%).**
   Using `delay_days = actual_delivery_date - promised_delivery_date`:
   - ~41% of shipments labeled "Delivered" were actually delivered late
   - ~44% of shipments labeled "Delayed" were actually on-time or early

   **Conclusion: all on-time/SLA analysis in this notebook is computed from dates,
   never from the `status` label.**

4. **682 rows marked "Delivered" or "Delayed" have no `actual_delivery_date` at all** —
   and (see below) every single one of them belongs to the South region. This isn't
   noise, it's a regional data pipeline gap. These rows are excluded from delay-based
   analysis rather than imputed, and South's on-time numbers are flagged as unreliable
   throughout this notebook.

5. **74 rows have `actual_delivery_date` earlier than `pickup_date` or `booking_date`** —
   logically impossible (can't be delivered before it was picked up). Excluded from
   delay-based analysis.

6. Minor: 71/87 rows missing `booking_date`/`pickup_date` (doesn't affect delay math).
   244 rows have identical origin/destination city (plausibly legitimate, just noted).

**Net effect: only 3,444 of 5,000 rows (69%) are usable for delay-based analysis.**
All Q1–Q3 analysis below uses this filtered `valid_for_delay_analysis` subset.


In [3]:
valid = df[df['valid_for_delay_analysis']]
print(f"Valid rows for delay analysis: {len(valid)} / {len(df)} ({len(valid)/len(df)*100:.1f}%)")


Valid rows for delay analysis: 3444 / 5000 (68.9%)


---
## Q1. Which region has the worst on-time delivery performance, and what's driving it?


In [4]:
# South must be checked separately first - is its data even usable?
south_check = df[df['region']=='South']
south_completed = south_check[south_check['status'].isin(['Delivered','Delayed'])]
print("South: completed shipments missing actual_delivery_date:",
      south_completed['actual_delivery_date'].isna().sum(), "/", len(south_completed))

north_completed = df[(df['region']=='North') & (df['status'].isin(['Delivered','Delayed']))]
print("North (comparison): completed shipments missing actual_delivery_date:",
      north_completed['actual_delivery_date'].isna().sum(), "/", len(north_completed))


South: completed shipments missing actual_delivery_date: 682 / 807
North (comparison): completed shipments missing actual_delivery_date: 0 / 831


In [5]:
# South has an 84% data gap on completed shipments - its on-time % (computed from
# only 124 surviving rows) is NOT comparable to other regions. Exclude from ranking.
reliable = valid[valid['region'] != 'South']

region_summary = reliable.groupby('region').agg(
    n=('shipment_id', 'count'),
    on_time_pct=('delay_days', lambda x: round((x <= 0).mean() * 100, 1)),
    breach_pct=('delay_days', lambda x: round((x > 0).mean() * 100, 1)),
    avg_delay_days=('delay_days', 'mean'),
).sort_values('breach_pct', ascending=False)
region_summary


,n,on_time_pct,breach_pct,avg_delay_days
region,,,,
Central,836,48.3,51.7,0.552632
North,811,49.6,50.4,0.574599
East,819,50.3,49.7,0.420024
West,854,51.3,48.7,0.272834


In [6]:
# Is this spread statistically meaningful, or within noise?
ct = pd.crosstab(reliable['region'], reliable['delay_days'] > 0)
chi2, p, dof, exp = chi2_contingency(ct)
print(f"Chi-square test (region vs breach): chi2={chi2:.2f}, p-value={p:.4f}")
print("-> Not significant at p<0.05: cannot claim any region is genuinely worse.")


Chi-square test (region vs breach): chi2=1.58, p-value=0.6648
-> Not significant at p<0.05: cannot claim any region is genuinely worse.


In [7]:
# Drill into Central (nominal worst) to look for a driver anyway
central = reliable[reliable['region'] == 'Central']
print("Central breach % by carrier:")
print(central.groupby('carrier_id').agg(
    n=('shipment_id', 'count'),
    breach_pct=('delay_days', lambda x: round((x > 0).mean() * 100, 1))
).sort_values('breach_pct', ascending=False))


Central breach % by carrier:
             n  breach_pct
carrier_id                
CARR_08     53        62.3
CARR_07     36        61.1
CARR_02     69        59.4
CARR_13     58        55.2
CARR_10     53        54.7
CARR_03     65        53.8
CARR_09     57        52.6
CARR_04     44        52.3
CARR_06     48        52.1
CARR_01     61        50.8
CARR_15     58        50.0
CARR_14     54        46.3
CARR_05     58        44.8
CARR_11     69        42.0
CARR_12     53        41.5


In [8]:
# Are 'bad' carriers concentrated in Central, or spread evenly across all regions?
carrier_region_share = pd.crosstab(reliable['carrier_id'], reliable['region'], normalize='index').round(3) * 100
print("% of each carrier's shipments that happen to be in Central:")
print(carrier_region_share['Central'].sort_values(ascending=False))
print()
print("Overall carrier breach % (all reliable regions combined):")
print(reliable.groupby('carrier_id').agg(
    n=('shipment_id', 'count'),
    breach_pct=('delay_days', lambda x: round((x > 0).mean() * 100, 1))
).sort_values('breach_pct', ascending=False))


% of each carrier's shipments that happen to be in Central:
carrier_id
CARR_03    30.8
CARR_02    30.0
CARR_11    27.8
CARR_12    26.8
CARR_13    26.2
CARR_01    25.8
CARR_14    25.4
CARR_05    25.0
CARR_09    24.7
CARR_15    24.4
CARR_08    24.2
CARR_06    24.1
CARR_10    24.0
CARR_04    21.2
CARR_07    16.7
Name: Central, dtype: float64

Overall carrier breach % (all reliable regions combined):
              n  breach_pct
carrier_id                 
CARR_02     230        59.1
CARR_07     215        55.8
CARR_13     221        55.7
CARR_08     219        53.9
CARR_06     199        51.8
CARR_03     211        50.7
CARR_04     208        50.5
CARR_01     236        50.0
CARR_05     232        49.6
CARR_09     231        48.1
CARR_10     221        46.6
CARR_12     198        46.0
CARR_15     238        45.8
CARR_14     213        45.1
CARR_11     248        44.0


### Q1 — Answer

**Region is not where the SLA problem lives — the real driver is carrier.**

- **South cannot be evaluated at all.** 84% of its "completed" shipments have no
  logged delivery date. Its apparent on-time rate is computed from a tiny,
  non-representative sample of 124 shipments and should not be trusted or compared.
  This is itself the most urgent finding: South's delivery-tracking pipeline is broken.
- Among the four regions with trustworthy data, breach rates range narrowly from
  48.7% (West) to 51.7% (Central) — and a chi-square test confirms this difference
  is **not statistically significant** (p = 0.66). We cannot claim any region is
  genuinely worse than another from this data.
- Mode mix and average distance are nearly identical across regions, so there's no
  structural reason to expect regional differences either.
- The real, robust signal is **carrier**: breach rates range from 44% (CARR_11) to
  59% (CARR_02) — a 15-point spread, five times wider than the regional spread —
  and this pattern holds consistently regardless of region (bad carriers aren't
  concentrated in any one place).

**Caveat:** `region` in this dataset is derived from the shipment's *origin* city,
not the destination — worth stating explicitly since "region" could be misread as
a delivery-side attribute.


---
## Q2. Is there a relationship between freight cost and distance? Which carrier(s) deviate, and by how much?


In [9]:
print("Overall correlation (all carriers):", round(df['freight_cost'].corr(df['distance_km']), 3))
for m in df['mode'].unique():
    sub = df[df['mode'] == m]
    print(f"  {m}: r = {sub['freight_cost'].corr(sub['distance_km']):.3f}")


Overall correlation (all carriers): 0.296
  PTL: r = 0.312
  FTL: r = 0.337
  LTL: r = 0.330


In [10]:
# Check cost-per-km outliers by mode before fitting anything
df['cost_per_km'] = df['freight_cost'] / df['distance_km']
print(df.groupby('mode')['cost_per_km'].describe())


       count       mean        std        min        25%        50%        75%         max
mode                                                                                      
FTL   2033.0  39.331341  55.704469  21.250794  23.215708  25.335782  27.194191  335.781630
LTL   1982.0  19.298762  27.111491  10.203923  11.178144  12.145412  13.080899  158.563827
PTL    985.0  13.332979  19.125842   6.803841   7.402143   8.026900   8.674733  107.049312


In [11]:
# Fit per-mode linear regression (cost structures differ meaningfully by mode)
df['predicted_cost'] = np.nan
for m in df['mode'].unique():
    mask = df['mode'] == m
    sub = df[mask]
    slope, intercept = np.polyfit(sub['distance_km'], sub['freight_cost'], 1)
    df.loc[mask, 'predicted_cost'] = intercept + slope * sub['distance_km']
    print(f"{m}: cost = {intercept:.1f} + {slope:.2f} * distance_km")

df['pct_deviation'] = (df['freight_cost'] - df['predicted_cost']) / df['predicted_cost'] * 100


PTL: cost = 153.3 + 13.07 * distance_km
FTL: cost = -2592.1 + 41.98 * distance_km
LTL: cost = -155.6 + 19.23 * distance_km


In [12]:
carrier_dev = df.groupby('carrier_id').agg(
    n=('shipment_id', 'count'),
    avg_pct_deviation=('pct_deviation', 'mean'),
).sort_values('avg_pct_deviation', ascending=False)
carrier_dev.round(1)


,n,avg_pct_deviation
carrier_id,,
CARR_07,342,521.0
CARR_05,329,-31.7
CARR_08,316,-33.3
CARR_13,309,-34.6
CARR_06,320,-34.7
CARR_15,361,-34.9
CARR_01,352,-35.4
CARR_10,330,-35.6
CARR_03,321,-36.4


**CARR_07 is a massive outlier** (avg +521% deviation vs everyone else in a tight
-30% to -57% band) — and because OLS is sensitive to outliers, CARR_07's extreme
values are dragging the fitted line up, making every *other* carrier look artificially
"underpriced" by comparison. Let's refit excluding CARR_07 to see the real picture.


In [13]:
carr07 = df[df['carrier_id'] == 'CARR_07']
others = df[df['carrier_id'] != 'CARR_07']

mode_median_cpk = others.groupby('mode')['cost_per_km'].median()
carr07 = carr07.copy()
carr07['ratio_to_median'] = carr07['cost_per_km'] / carr07['mode'].map(mode_median_cpk)
print("CARR_07 cost-per-km as a multiple of the normal median, by shipment:")
print(carr07['ratio_to_median'].describe())
print()
print("Shipments with ratio > 3x normal median:", (carr07['ratio_to_median'] > 3).sum(), "/", len(carr07))


CARR_07 cost-per-km as a multiple of the normal median, by shipment:
count    342.000000
mean       9.893905
std        1.438573
min        6.949119
25%        8.805021
50%        9.825085
75%       10.864547
max       13.522123
Name: ratio_to_median, dtype: float64

Shipments with ratio > 3x normal median: 342 / 342


In [14]:
# Refit the model excluding CARR_07 entirely
clean = df[df['carrier_id'] != 'CARR_07'].copy()
clean['predicted_cost'] = np.nan
for m in clean['mode'].unique():
    mask = clean['mode'] == m
    sub = clean[mask]
    slope, intercept = np.polyfit(sub['distance_km'], sub['freight_cost'], 1)
    clean.loc[mask, 'predicted_cost'] = intercept + slope * sub['distance_km']
    print(f"{m}: cost = {intercept:.1f} + {slope:.2f} * distance_km  (r={sub['freight_cost'].corr(sub['distance_km']):.3f})")

clean['pct_deviation'] = (clean['freight_cost'] - clean['predicted_cost']) / clean['predicted_cost'] * 100
print()
print(clean.groupby('carrier_id').agg(
    n=('shipment_id', 'count'),
    avg_pct_dev=('pct_deviation', 'mean')
).sort_values('avg_pct_dev', ascending=False).round(1))


FTL: cost = 0.4 + 25.01 * distance_km  (r=0.985)
LTL: cost = -14.4 + 12.03 * distance_km  (r=0.985)
PTL: cost = 51.3 + 7.90 * distance_km  (r=0.984)

              n  avg_pct_dev
carrier_id                  
CARR_05     329          1.0
CARR_11     362          0.3
CARR_01     352          0.2
CARR_12     312          0.1
CARR_09     344          0.1
CARR_14     323         -0.1
CARR_06     320         -0.1
CARR_04     319         -0.2
CARR_02     360         -0.2
CARR_03     321         -0.3
CARR_15     361         -0.3
CARR_08     316         -0.3
CARR_10     330         -0.8
CARR_13     309         -1.1


### Q2 — Answer

- Freight cost has a **near-perfect linear relationship** with distance once you
  separate by mode: FTL ≈ ₹25/km, LTL ≈ ₹12/km, PTL ≈ ₹8/km, with **r = 0.985**
  for all three modes (after removing one outlier carrier — see below).
- **CARR_07 is the one true anomaly.** All 342 of its shipments — 100%, not a
  subset — bill at roughly 7–13x (avg ~10x) the rate every other carrier charges
  for the same mode and distance. The tightness of that multiple (mean 9.9x,
  std only 1.4) points to a **systematic issue** — a cost field off by a factor
  of ~10, or a currency/unit mismatch — rather than normal pricing variance.
  Worth verifying against the billing source before treating it as real cost data.
- Every other carrier (14 of 15) prices within **~1%** of the expected cost curve —
  effectively no deviation at all once CARR_07 is set aside.


---
## Q3. Which customer(s) show the most delivery delays — carrier-driven, region-driven, or something else?


In [15]:
cust_summary = valid.groupby('customer_id').agg(
    n=('shipment_id', 'count'),
    n_breach=('delay_days', lambda x: (x > 0).sum()),
).reset_index()
cust_summary['breach_pct'] = (cust_summary['n_breach'] / cust_summary['n'] * 100).round(1)
print(f"{cust_summary['n'].shape[0]} unique customers, {cust_summary['n'].mean():.0f} shipments each on average")
cust_summary.sort_values('breach_pct', ascending=False).head(10)


120 unique customers, 29 shipments each on average


,customer_id,n,n_breach,breach_pct
25,CUST_026,23,17,73.9
49,CUST_050,31,22,71.0
115,CUST_116,34,24,70.6
62,CUST_063,33,23,69.7
118,CUST_119,32,22,68.8
113,CUST_114,32,22,68.8
70,CUST_071,27,18,66.7
61,CUST_062,23,15,65.2
16,CUST_017,33,21,63.6
50,CUST_051,22,14,63.6


In [16]:
# Before naming any customer as a real problem: is this spread just noise,
# given we're testing 120 customers with small samples each (~20-35 shipments)?
overall_rate = (valid['delay_days'] > 0).mean()

def pval(row):
    return binomtest(row['n_breach'], row['n'], overall_rate, alternative='two-sided').pvalue

cust_summary['p_value'] = cust_summary.apply(pval, axis=1)
sig = cust_summary[cust_summary['p_value'] < 0.05]
print(f"Customers significant at p<0.05: {len(sig)} out of {len(cust_summary)}")
print("(Expected by chance alone at this threshold: ~", round(len(cust_summary)*0.05, 1), ")")
sig.sort_values('breach_pct', ascending=False)


Customers significant at p<0.05: 6 out of 120
(Expected by chance alone at this threshold: ~ 6.0 )


,customer_id,n,n_breach,breach_pct,p_value
25,CUST_026,23,17,73.9,0.034690
49,CUST_050,31,22,71.0,0.029450
115,CUST_116,34,24,70.6,0.024307
62,CUST_063,33,23,69.7,0.035083
58,CUST_059,36,12,33.3,0.046905
47,CUST_048,39,10,25.6,0.002209


In [17]:
# Even though not statistically robust, check if the top raw-rate customers
# share a concentrated carrier or region (which would still be actionable)
top4 = ['CUST_026', 'CUST_050', 'CUST_116', 'CUST_063']
sub = valid[valid['customer_id'].isin(top4)]
print("Region mix for top-4 customers by raw breach%:")
print(pd.crosstab(sub['customer_id'], sub['region']))
print()
print("% of shipments via CARR_02 (worst 'normal' carrier), vs baseline:")
baseline = (valid['carrier_id'] == 'CARR_02').mean() * 100
for c in top4:
    pct = (valid[valid['customer_id']==c]['carrier_id'] == 'CARR_02').mean() * 100
    print(f"  {c}: {pct:.1f}%  (baseline: {baseline:.1f}%)")


Region mix for top-4 customers by raw breach%:
region       Central  East  North  South  West
customer_id                                   
CUST_026           5     5      5      0     8
CUST_050          12     6      3      2     8
CUST_063           5    11      8      1     8
CUST_116           7     4     12      2     9

% of shipments via CARR_02 (worst 'normal' carrier), vs baseline:


  CUST_026: 8.7%  (baseline: 7.0%)
  CUST_050: 9.7%  (baseline: 7.0%)
  CUST_116: 11.8%  (baseline: 7.0%)
  CUST_063: 12.1%  (baseline: 7.0%)


### Q3 — Answer

- **No individual customer is a statistically genuine outlier.** Out of 120
  customers tested, ~6 come back "significant" at p<0.05 — almost exactly what
  you'd expect from random chance alone when testing 120 groups at that threshold.
  That's the textbook signature of *no real effect*, not a true finding.
- The customers with the highest raw breach rates (66-74%) don't share a
  concentrated carrier or region either — their shipment mix looks close to
  baseline proportions across the board.
- **Conclusion: it's neither carrier-driven nor region-driven for these specific
  customers — it's most consistent with sampling noise**, since each customer
  only has 20-35 shipments in this dataset. The real, robust levers remain what
  Q1/Q2 found: CARR_02's consistently elevated breach rate, and CARR_07's cost
  anomaly.
- **Recommendation:** don't act on "problem customers" from a single 5,000-row
  snapshot. Track customer-level breach rate on a rolling basis over more
  shipments before treating any one customer as a real issue.


---
## Q4. Data quality issues found, and how they were handled

See the full report generated at the top of this notebook (`quality_report`), and the
detailed writeup in `docs/BUSINESS_ANSWERS.md`. Summary of what was found and the handling
decision for each:

| Issue | Rows affected | Handling |
|---|---|---|
| Exact duplicate rows | 15 | Dropped |
| `delivery_date` redundant with `promised_delivery_date` (100% match) | all 5,000 | Ignored column entirely; used `actual_delivery_date` as ground truth |
| `status` disagrees with date-derived delay | 1,742 (35%) | All delay/SLA metrics computed from dates, never from `status` |
| Completed status but missing `actual_delivery_date` (100% concentrated in South) | 682 | Excluded from delay analysis; flagged as a regional data pipeline issue |
| `actual_delivery_date` before `pickup_date`/`booking_date` (impossible) | 74 | Excluded from delay analysis |
| Missing `booking_date` / `pickup_date` | 71 / 87 | Left as-is, doesn't affect delay calculations |
| Origin city == destination city | 244 | Kept, plausibly legitimate, just noted |


---
## Q5. One metric to track weekly to catch delivery problems early

**On-time delivery rate, computed strictly from dates (`actual_delivery_date` vs
`promised_delivery_date`, never from `status`), broken out by carrier rather than
blended into one company-wide number** — because Q1/Q3 showed the carrier effect is
real and large (15pp spread) while region and customer effects were statistically
indistinguishable from noise. A blended company-wide number would hide exactly the
signal that's actionable.

**Mandatory paired guardrail:** % of shipments missing a logged `actual_delivery_date`
within N days of their promised date. Without this check, a metric can quietly become
meaningless — exactly what happened to South, where the underlying data pipeline broke
and nothing would have caught it without watching completeness directly.
